# Cross-domain single-class pothole detector (Component 10.4B)

YOLO11s trained on the pooled 4-domain single-class set (far/dashcam + close-up + tiny/far + mid).
**Recipe is the Component-3 baseline, held fixed — only the data and `nc=1` differ** (for a fair
single-dataset-vs-cross-domain comparison). Resumable; checkpoints to Drive every epoch. Test-set
eval is NOT here — that's 10.5 with the stratified evaluator.


## Cell 1 - clone/pull, install, GPU assert

In [ ]:
import os
REPO_URL = "https://github.com/YashChugani/adaptive-tile-pothole-detection.git"  # edit if needed (PAT for private)
REPO_DIR = "/content/Pothole-Detection-System"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
# Install ONLY ultralytics+sahi; keep Colab's GPU torch (pinning torch risks CUDA mismatch -> silent CPU).
!pip install -q ultralytics==8.4.160 sahi==0.12.6
%cd {REPO_DIR}
import torch
assert torch.cuda.is_available(), "No CUDA GPU. Runtime > Change runtime type > GPU."
print("torch", torch.__version__, "| CUDA GPU:", torch.cuda.get_device_name(0))


## Cell 2 - setup (Drive mount, pkg -> local SSD, runtime yaml, Arial, checkpoints)

In [ ]:
import sys
sys.path.insert(0, "/content/Pothole-Detection-System/notebooks")
import colab_setup_crossdomain as cs
cfg = cs.setup()
cfg


## Cell 3 - train (resumable)

Baseline recipe fixed: yolo11s.pt, imgsz 640, 100 epochs, batch 16, **seed 0** (== baseline default,
for comparability), default augmentation. `save_period=1` persists every epoch to Drive. Auto-resumes
from `last.pt` if present. Checkpoint dir is `checkpoints/crossdomain/yolo11s_crossdomain` — **separate
from baseline's `checkpoints/baseline/`, no collision**.

In [ ]:
import time, shutil
from pathlib import Path
from ultralytics import YOLO

# ---- recipe (fixed; only data + nc=1 differ from Component 3 baseline) ----
DATA    = cfg["data_yaml"]
PROJECT = cfg["project"]                 # MyDrive/pothole/checkpoints/crossdomain
NAME    = "yolo11s_crossdomain"
IMGSZ, EPOCHS, BATCH, DEVICE, SEED = 640, 100, 16, 0, 0   # seed 0 == baseline default
FIRST_EPOCH_MAX_SEC = 900                # tripwire: >15 min/epoch on a T4 == likely silent CPU
# --------------------------------------------------------------------------

last_ckpt = Path(PROJECT) / NAME / "weights" / "last.pt"

# epoch-time backstop + silent-CPU tripwire: fail LOUD on epoch 1 instead of wasting hours.
_t = {"start": None, "last": None, "tripped": False}
def _epoch_timer(trainer):
    now = time.time()
    ref = _t["last"] if _t["last"] is not None else _t["start"]
    dur = (now - ref) if ref is not None else None
    if dur is not None and trainer.epoch < 5:
        print(f"[backstop] epoch {trainer.epoch} wall-time {dur:.1f}s")
    if dur is not None and not _t["tripped"]:           # check the FIRST completed epoch (fresh or resume)
        _t["tripped"] = True
        if dur > FIRST_EPOCH_MAX_SEC:
            raise RuntimeError(f"first epoch took {dur:.0f}s > {FIRST_EPOCH_MAX_SEC}s "
                               f"-- likely silent CPU fallback, check device")
    _t["last"] = now

# pick model (resume vs fresh), wire the callback + start-timer ONCE, then train on either path
if last_ckpt.exists():
    print(f"RESUMING from {last_ckpt}")
    model = YOLO(str(last_ckpt))
else:
    print("FRESH start from yolo11s.pt")
    model = YOLO("yolo11s.pt")
model.add_callback("on_fit_epoch_end", _epoch_timer)
_t["start"] = time.time()

if last_ckpt.exists():
    results = model.train(resume=True)
else:
    results = model.train(
        data=DATA, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=DEVICE, seed=SEED,
        project=PROJECT, name=NAME, save=True, save_period=1, exist_ok=True, resume=False,
    )

# runs on BOTH paths: persist best.pt to the checkpoint-dir root + print VAL metrics (no test eval -> 10.5)
best = Path(PROJECT) / NAME / "weights" / "best.pt"
shutil.copy(best, Path(PROJECT) / "best.pt")
print("best.pt ->", Path(PROJECT) / "best.pt")
print("VAL metrics:", results.results_dict)
